## Proyecto programado - Electro II
#### Manfred Alvarado López - C10318 ; Anaité Chaves Ramirez - C12154

Este notebook corresponde al proyecto programado 1 del curso Electromagnetismo II. 

In [1]:
# Si no esta instalado hay que instalar el paquete de ipwidgets y reiniciar el kernel
# %pip install ipywidgets
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML, display

In [2]:
plt.rcParams['animation.embed_limit'] = 200 # Limite en MB para que no se corte la animación
c = 1.0 # Velocidad de la luz normalizada
NF = 36 # Número de frames

def campos(x, z, t , om, a):
    """
    Calcula las amplitudes reales de los campos electromagnéticos para el modo TE10. Evalúa si la frecuencia dada permite la propagación de la onda o si decae de forma evanescente.
    
    Parámetros:
    x : float o numpy.darray - Coordenada transversal (eje x) dentro de la guía.
    z : float o numpy.darray - Coordenada longitudinal (eje z) de propagación.
    t : float - Tiempo actual a evaluar.
    om : float - Frecuencia angular de la onda electromagnética.
    a : float - Ancho de la guía de onda (dimensión tranversal en x)
    
    Retorna:
    tuple : Contiene tres elementos (Ey, Bx, Bz) correspondientes a las amplitudes reales de los campos evaluados en los puntos (x, z) al instante t. 
    
    """
    
    wc = np.pi * c / a #Frecuencia de corte para el modo TE10
    
    # Amplitudes espaciales en el eje transversal
    s = np.sin(np.pi * x / a)
    co = np.cos(np.pi * x / a)
    
    if om > wc:
        # Régimen propagante (omega  > omega_c)
        kz = np.sqrt(om**2 - wc**2) / c # Número de onda longitudinal real
        ph = kz * z - om * t # Fase de la onda viajera
        
        # Se retorna Ey, Bx, Bz multiplicados por su oscilación espacial y temporal
        return s * np.sin(ph), -(kz / om) * s * np.sin(ph), -(np.pi / (a * om)) * co * np.cos(ph)
    
    else:
        # Régimen Evanescente (omega < omega_c)
        kap = np.sqrt(wc**2 - om**2) / 2 # Constante de atenuación kappa (número de onda imaginario)
        e = np.exp(-kap * z) # Decaimiento exponencial a lo largo del z
        
        # La onda oscila en el tiempo pero no viaja, su amplitud cae exponencialmente
        return (s * e * np.cos(om * t), -(kap / om) * s * e * np.sin(om * t), -(np.pi / (a * om)) * co * e * np.sin(om * t))
    
    
def generar(r=1.6, a=2.3, b=1.0):
    """
    Configura la interfaz gráfica, inicializa las mallas espaciales (grids) y estructura la figura que contiene la animación de los campos y las gráficas estáticas de dispersión.
    
    Parámetros:
    r : float - Razón entre la frecuencia angular dada y la frecuencia de corte (omega / omega_c).
    a : float - Ancho de la guía de onda en centímetros.
    b : float - Altura de la guía de onda en centímetros.
    
    Retorna:
    Ninguno. Despliega la animación en formato HTML directamente en el entorno interactivo.
    """
    
    # Cálculos iniciales de los parámetros de la onda
    wc = np.pi * c / a # Frecuencia de corte angular
    om = r * wc # Frecuencia angular de trabajo
    T = 2 * np.pi / om # Periodo de oscilación
    fc = 14.99 / a # Frecuencia de corte en GHz (con a en cm)
    r_sig = min(2.0, a / b) # Razón límite antes de excitar el siguiente modo (TE20 o TE01)
    
    # Configuración de la figura principal de Matplotlib
    fig = plt.figure(figsize=(10, 8))
    fig.suptitle(f'a = {a:.1f} cm, b = {b:.1f} cm  ;  fc = {fc:.2f} GHz,  f = {r*fc:.2f} GHz', y=0.98)
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.15], hspace=0.35, wspace=0.25)
    
    """
    Subplot 1: Vista superior de la guia (Animación 2D)
    
    """
    ax = fig.add_subplot(gs[0, :])
    
    # Malla densa para mapear el gradiente de color del campo eléctrico Ey
    z = np.linspace(0, 3 * a, 300)
    x = np.linspace(0, a, 100)
    Z, X = np.meshgrid(z, x)
    im = ax.imshow(campos(X, Z, 0, om, a)[0], extent=[0, 3 * a, 0, a], origin='lower', cmap='RdBu_r', vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label='Ey / E0')
    
    # Malla dispersa para graficar los vectores (flechas) del campo magnético B
    zq, xq = np.meshgrid(np.linspace(0.06, 2.94, 24) * a, np.linspace(0.1, 0.9, 5) * a)
    bn = max(1, 1 / r); L = 0.11 * a / bn
    _, Bx, Bz = campos(xq, zq, 0, om, a)
    Q = ax.quiver(zq, xq, Bz * L, Bx * L, angles='xy', scale_units='xy', scale=1, pivot='mid', width=0.002)
    
    ax.set_xlabel('z [cm]'); ax.set_ylabel('x [cm]')
    ax.set_title('Ey (color) y B en el plano xz (flechas) — paredes en x = 0 y x = a')
    
    """
    Subplot 2: Sección transversal de la guía (z = 0)
    
    """
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.add_patch(Rectangle((0, 0), a, b, fill=False, lw=3))
    
    # Vectores para representar cómo varía la intensidad del campo Ey a lo ancho de 'a'
    xs = np.linspace(0.06 * a, 0.94 * a, 11)
    Q2 = ax2.quiver(xs, np.full_like(xs, b / 2), 0 * xs, campos(xs, 0.0, 0, om, a)[0] * 0.8 * b, angles='xy', scale_units='xy', scale=1, pivot='mid', width=0.008)
    
    m = 0.1 * max(a, b); ax2.set_xlim(-m, a + m); ax2.set_ylim(-m, b + m); ax2.set_aspect('equal')
    ax2.set_xlabel('x [cm]'); ax2.set_ylabel('y [cm]')
    ax2.set_title('Sección transversal (z = 0): campo E (en y)')
    
    """
    Subplot 3: Gráfica de Dispersión estática (Relación de omega vs k_z)
    
    """
    ax3 = fig.add_subplot(gs[1, 1])
    k = np.linspace(0, 3, 300); kk = np.linspace(0, 1, 200)
    
    ax3.plot(k, np.sqrt(1 + k**2), label=r'$\omega=\sqrt{\omega_c^2+c^2k_z^2}$') # Dispersión teórica
    ax3.plot(-kk, np.sqrt(1 - kk**2), 'C3--', label=r'evanescente ($\kappa$)') # Rama de atenuación
    ax3.plot(k, k, 'k:', label=r'$\omega=ck$') # Cono de luz (vacío)
    ax3.axhline(1, color='gray') # Límite de corte
    ax3.axhline(r_sig, color='C1', ls='--', label='corte del siguiente modo')
    
    
    
    # Análisis cinemático de velocidades dependiente del régimen (propagante vs evanescente)
    if r > 1:
        kz_ = np.sqrt(r**2 - 1)
        vg = kz_ / r # Velocidad de grupo (vg < c)
        vf = r / kz_ # Velocidad de fase (vf > c)
        
        ax3.plot([0, kz_ * 1.4], [0, r * 1.4], label=r'pendiente $v_f/c$')
        ax3.plot([kz_ - .5, kz_ + .5], [r - .5 * vg, r + .5 * vg], label=r'pendiente $v_g/c$')
        ax3.plot(kz_, r, 'ko') # Punto operativo actual
        titulo = f'Propagante: $v_f$={vf:.2f}c (>c), $v_g$={vg:.2f}c, $v_f v_g$=c²'
    else:
        kap = np.sqrt(1 - r**2)
        ax3.plot(-kap, r, 'ko') # Punto operativo en régimen evanescente
        titulo = (f'Evanescente: κ={kap:.2f} ωc/c, penetración 1/κ=' + (f'{a/(np.pi*kap):.2f} cm' if r < 0.9999 else '∞'))


# Validaciones geométricas
    if b > a:
        titulo += '\n(b > a: el TE10 ya no es el modo fundamental)'
    elif r > r_sig:
        titulo += '\n(¡ya se propagan otros modos además del TE10!)'
        
    ax3.set_xlim(-1.1, 3); ax3.set_ylim(0, 3)
    ax3.set_xlabel(r'$k_z c/\omega_c$  (izq.: $\kappa c/\omega_c$)'); ax3.set_ylabel(r'$\omega/\omega_c$')
    ax3.set_title(titulo, fontsize=10); ax3.legend(fontsize=8, loc='upper left')
    
    # Renderizamos la figura en memoria para calcular los bordes exactos
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    
    # 1. Exporta la vista superior (Morfología y Atenuación)
    caja_ax1 = ax.get_tightbbox(rend).transformed(fig.dpi_scale_trans.inverted())
    
    # 2. Exporta la sección transversal (Metodología)
    caja_ax2 = ax2.get_tightbbox(rend).transformed(fig.dpi_scale_trans.inverted())
    
    # 3. Exporta la gráfica de dispersión (Cinemática y Causalidad)
    caja_ax3 = ax3.get_tightbbox(rend).transformed(fig.dpi_scale_trans.inverted())
    
    
    def update(i):
        """
        Función interna de actualización para Matplotlib FuncAnimation. Recalcula los campos electromagnéticos para un nuevo paso de tiempo discreto.
        
        Parámetros:
        i : int - Índice del frame actual de la animación (0 a NF - 1)
        
        Retorna:
        tuple : Elementos gráficas actualizados (imagen y conjuntos de flechas) para renderizado.
        """
        
        t = T * i / NF  # Calcula el tiempo físico correspondiente al frame 'i'
            
        # Actualiza los datos de color de la imagen (Campo Ey)
        im.set_data(campos(X, Z, t, om, a)[0])
            
        # Recalcula y actualiza los vectores del campo magnético (Bx, Bz)
        _, Bx, Bz = campos(xq, zq, t, om, a)
        Q.set_UVC(Bz * L, Bx * L)
            
        # Actualiza los vectores del campo eléctrico transversal en z=0
        Q2.set_UVC(0 * xs, campos(xs, 0.0, t, om, a)[0] * 0.8 * b)
            
        return im, Q, Q2

    # Construcción y visualización de la animación (DENTRO de generar)
    anim = FuncAnimation(fig, update, frames=NF, interval=60, blit=False)
    plt.close(fig) # Evita que se duplique una figura estática en el output de Jupyter
    display(HTML(anim.to_jshtml(default_mode='loop')))

# Interfaz interactiva: mueve los deslizadores y pulsa "Run Interact" para regenerar
w.interact_manual(generar,
    r=w.FloatSlider(value=1.6, min=0.2, max=3, step=0.01, description='ω/ωc'),
    a=w.FloatSlider(value=2.3, min=1, max=10, step=0.1, description='a [cm]'),
    b=w.FloatSlider(value=1.0, min=0.3, max=10, step=0.1, description='b [cm]'))

interactive(children=(FloatSlider(value=1.6, description='ω/ωc', max=3.0, min=0.2, step=0.01), FloatSlider(val…

<function __main__.generar(r=1.6, a=2.3, b=1.0)>